In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from time import sleep

## Deputados e Suplentes no Momento (SNAPSHOT)

In [ ]:
url = "https://dadosabertos.camara.leg.br/api/v2/deputados?itens=100"

all_data = []

while url:
    res = requests.get(url).json()
    
    all_data.extend(res['dados'])
    
    # pega próximo link
    next_link = [l['href'] for l in res['links'] if l['rel'] == 'next']

    print(f"Próximo link: {next_link}")

    url = next_link[0] if next_link else None


print(f"\nTotal de deputados coletados: {len(all_data)}")

In [ ]:
df = pd.DataFrame(all_data)

print(df.info())

df.head()

In [ ]:
df.to_csv('deputados_2026.csv', index=False)

## Deputados e Suplentes no Período

In [ ]:
url2 = "https://dadosabertos.camara.leg.br/api/v2/deputados?dataInicio=2023-02-01&dataFim=2026-04-30&itens=100"

all_data2 = []

while url2:
    res = requests.get(url2).json()
    
    all_data2.extend(res['dados'])
    
    # pega próximo link
    next_link = [l['href'] for l in res['links'] if l['rel'] == 'next']

    print(f"Próximo link: {next_link}")

    url2 = next_link[0] if next_link else None

print(f"\nTotal de deputados coletados: {len(all_data2)}")

In [ ]:
df2 = pd.DataFrame(all_data2)

print(df2.info())

df2.head()

In [ ]:
df2_agrupado_contagem_partidos_por_politicos = df2.groupby(['nome', 'id'])['id'].count().rename('contagem').reset_index().sort_values(by='contagem', ascending=False)
df2_agrupado_contagem_partidos_por_politicos[df2_agrupado_contagem_partidos_por_politicos['contagem'] > 1]

#### Retirando os duplicados por Eventos

In [ ]:
df2_politicos_unicos = df2.drop_duplicates(subset='id', keep='first')
df2_politicos_unicos.to_csv('deputados_2026_id_legis_57.csv', index=False)

### Sinalizar os deputados que estão atuando no momento x Histórico

In [ ]:
df_merge = df2_politicos_unicos.merge(
    df[['id', 'nome', 'siglaPartido', 'siglaUf']],
    on='id',
    how='left',
    suffixes=('_hist', '_atual'),
    indicator=True
)

In [ ]:
df_merge.head()

In [ ]:
def classificar(row):
    if row['_merge'] == 'left_only':
        return 'não ativo (ex-deputado,suplente,suspenso, etc)'
    else:
        return 'ativo'

df_merge['status'] = df_merge.apply(classificar, axis=1)

In [ ]:
df_merge.head()

### Sinalizar Mudanças de Partido entre os Deputados

In [ ]:
df3_mudanca_partido = df2.groupby('id')['siglaPartido'].nunique().reset_index()

df3_mudanca_partido['mudou_partido'] = df3_mudanca_partido['siglaPartido'] > 1

In [ ]:
df3_mudanca_partido.head()

In [ ]:
df_final_merge = df_merge.merge(df3_mudanca_partido[['id', 'mudou_partido']], on='id', how='left')

In [ ]:
def classificar_partido(row):
    if row['status'] != 'ativo':
        if row['mudou_partido']:
            return 'mudou/saiu'
        else:
            return 'não mudou/não ativo'
    else:
        if row['mudou_partido']:
            return 'mudou/ativo'
        else:
            return 'estável'

df_final_merge['status_partido'] = df_final_merge.apply(classificar_partido, axis=1)

In [ ]:
df_final_merge

### Filtros na Planilha Final

In [ ]:
df_deputados_ativos = df_final_merge[df_final_merge['status']=='ativo']
df_deputados_ativos

In [ ]:
df_deputados_nao_ativos = df_final_merge[df_final_merge['status']!='ativo']
df_deputados_nao_ativos

### Consumindo dados de despesas

In [ ]:
df_id_deputados_ativos = df_deputados_ativos['id']
print(df_id_deputados_ativos.info())
df_id_deputados_ativos.head()

In [ ]:
len(df_id_deputados_ativos)

#### Arrumar essa parte depois

In [ ]:
backup_01 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_25.csv')
backup_02 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_50.csv')
backup_03 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_75.csv')
backup_04 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_100.csv')
backup_05 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_125.csv')
backup_06 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_150.csv')
backup_07 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_175.csv')
backup_08 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_200.csv')
backup_09 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_225.csv')
backup_10 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/backup_despesas_faltantes_250.csv')


In [ ]:
backup_merged = pd.concat([backup_01, backup_02, backup_03, backup_04, backup_05, backup_06, backup_07, backup_08, backup_09, backup_10], ignore_index=True)

In [ ]:
backup_merged

In [ ]:
backup_merged.to_csv('/home/eduardo/documentos/pipelines_dados_politica/merged_despesas_151_250.csv', index=False)

In [ ]:
backup_merged_02 = pd.read_csv('/home/eduardo/documentos/pipelines_dados_politica/merged_backup_despesas_150.csv')

In [ ]:
merged_backups = pd.concat([backup_merged_02, backup_merged], ignore_index=True)

In [ ]:
df_deputados_ativos_faltantes = df_deputados_ativos[~df_deputados_ativos['id'].isin(merged_backups['id_deputado'])]

In [ ]:
base_url3 = "https://dadosabertos.camara.leg.br/api/v2/deputados/{id}/despesas"

lista_despesas_deputados = []
backup_lote = []

contagem_deputados_total = 0

for deputado_id in df_deputados_ativos_faltantes['id']:

    url3_deputado = base_url3.format(id=deputado_id)
    parametros = {
        'idLegislatura': 57,
        'ordem': 'ASC',
        'ordenarPor': 'ano'    
    }
    
    while url3_deputado:

        response = requests.get(url3_deputado, params=parametros)

        if response.status_code != 200:

            print(f"Erro ao acessar {url3_deputado}: {response.status_code}")
            break

        data = response.json()


        print(f"Coletando despesas do deputado {deputado_id} - \nURL: {url3_deputado}")

        # Despesas
        
        despesas = data.get('dados', [])

        if not despesas:
            print(f"Deputado {deputado_id} sem despesas nessa página")

        if despesas:
            df3_temporario = pd.DataFrame(despesas)
            df3_temporario['id_deputado'] = deputado_id

            lista_despesas_deputados.append(df3_temporario)
            backup_lote.append(df3_temporario)  #


        # Próximo link

        links = data.get('links', [])
        next_url = None

        for link in links:
            if link.get('rel') == 'next':
                next_url = link.get('href')

        url3_deputado = next_url
        parametros = None

        print(f"Próximo link para deputado {deputado_id}: \n{url3_deputado}")

        sleep(0.2)  # Evita sobrecarregar o servidor

    contagem_deputados_total += 1
    print(f"Total de deputados processados até agora: {contagem_deputados_total}")
    print(f"Tamanho atual do backup_lote: {len(backup_lote)}")

        # NOVO: salva a cada 25 deputados
    if contagem_deputados_total % 25 == 0 and backup_lote:
        df_backup = pd.concat(backup_lote, ignore_index=True)
        df_backup.to_csv(f'backup_despesas_faltantes_02_{contagem_deputados_total}.csv', index=False)
        print(f"Backup salvo com {contagem_deputados_total} deputados!")

        backup_lote = []  # limpa o lote

# FINAL (caso sobre resto < 25)
if backup_lote:
    df_backup = pd.concat(backup_lote, ignore_index=True)
    df_backup.to_csv(f'backup_despesas_final.csv', index=False)
                

# Junta tudo
df_final = pd.concat(lista_despesas_deputados, ignore_index=True)



In [ ]:
df_final.to_csv('merged_backup_251_513.csv', index=False)

In [ ]:
df_4 = pd.read_csv('merged_backup_251_513.csv')

In [ ]:
merged_despesas_final = pd.concat([merged_backups, df_4], ignore_index=True)

In [ ]:
(len(merged_despesas_final['id_deputado'].unique()))

In [ ]:
merged_despesas_final['id_deputado'].unique()

In [ ]:
df_deputados_ativos_faltantes = df_deputados_ativos[df_deputados_ativos['id'].isin(merged_despesas_final['id_deputado'])]


In [ ]:
merged_despesas_final.to_csv('merged_despesas_final_completa.csv', index=False)

In [ ]:
merged_despesas_final